In [11]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# authenticate
credential = DefaultAzureCredential()

# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id="abb3353d-14ab-4405-8fec-2be226eecc91",
    resource_group_name="BARRIBAL.JOSHUAGEORGE-rg",
    workspace_name="aquagrow",
)

In [25]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
import time

# update the 'my_path' variable to match the location of where you downloaded the data on your
# local filesystem

data_path = "./data/meanIntervalData.csv"
# set the version number of the data asset to the current UTC time
v1_m = time.strftime("%Y.%m.%d.%H%M%S", time.gmtime())


mean_data = Data(
    name="means-dataset",
    version=v1_m,
    description="Aquagrow mean data set",
    path=data_path,
    type=AssetTypes.URI_FILE,
)

# create data asset
ml_client.data.create_or_update(mean_data)

print(f"Data asset created. Name: {mean_data.name}, version: {mean_data.version}")

Data asset created. Name: means-dataset, version: 2023.05.04.183453


In [26]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
import time

# update the 'my_path' variable to match the location of where you downloaded the data on your
# local filesystem

weight_path = "./data/lettuceWeights.csv"
# set the version number of the data asset to the current UTC time
v1_w = time.strftime("%Y.%m.%d.%H%M%S", time.gmtime())


weight_data = Data(
    name="weights-dataset",
    version=v1_w,
    description="Aquagrow observed lettuce weights",
    path=weight_path,
    type=AssetTypes.URI_FILE,
)

# create data asset
ml_client.data.create_or_update(weight_data)

print(f"Data asset created. Name: {weight_data.name}, version: {weight_data.version}")

Data asset created. Name: weights-dataset, version: 2023.05.04.183457


In [ ]:
%pip install -U azureml-fsspec

In [29]:
import pandas as pd

# get a handle of the data asset and print the URI
mean_asset = ml_client.data.get(name="means-dataset", version=v1_m)
print(f"Data asset URI: {mean_asset.path}")

weight_asset = ml_client.data.get(name="weights-dataset", version=v1_w)
print(f"Data asset URI: {weight_asset.path}")

Data asset URI: azureml://subscriptions/abb3353d-14ab-4405-8fec-2be226eecc91/resourcegroups/BARRIBAL.JOSHUAGEORGE-rg/workspaces/aquagrow/datastores/workspaceblobstore/paths/LocalUpload/7251c980c9ce25d1bc38a162a9dbd6f8/meanIntervalData.csv
Data asset URI: azureml://subscriptions/abb3353d-14ab-4405-8fec-2be226eecc91/resourcegroups/BARRIBAL.JOSHUAGEORGE-rg/workspaces/aquagrow/datastores/workspaceblobstore/paths/LocalUpload/47051962db5bffde1f407af34f34a61b/lettuceWeights.csv


In [30]:
# read into pandas - note that you will see 2 headers in your data frame - that is ok, for now
mean_dataset = pd.read_csv(mean_asset.path)
mean_dataset.head()

,ambient_temperature,humidity,dissolved_oxygen,electrical_conductivity,ph_level,temperature
0,34.896598,62.753868,6.307200,9.256473,6.984857,30.950521
1,34.362855,64.693197,6.748143,9.198907,7.089195,30.433486
2,31.620019,73.221110,7.264633,9.026523,7.563556,28.250383
3,30.304183,76.367711,7.208256,8.451398,8.035779,26.484686
4,30.360142,75.203441,7.075773,8.008163,8.565331,26.515291


In [31]:
# read into pandas - note that you will see 2 headers in your data frame - that is ok, for now
weight_dataset = pd.read_csv(weight_asset.path)
weight_dataset.head()

,Week,Weight
0,0,13.500000
1,0,8.600000
2,0,10.500000
3,0,10.866667
4,1,32.600000


In [39]:
#create a new dataset
new_ds = []

# Loop through each row in the first DataFrame
for i, row in mean_dataset.iterrows():
    # Loop through each row in the second DataFrame
    for j, matching_row in weight_dataset.iterrows():
        # Create a copy of the current row from the first DataFrame
        combined_row = row.copy()
        # Add the data from the current row in the second DataFrame to the copy of the first DataFrame
        for column in weight_dataset.columns:
            if column != 'key':
                combined_row[column] = matching_row[column]
        # Append the combined row to the list of data frames
        new_ds.append(pd.DataFrame([combined_row]))

# Concatenate the list of data frames into a single data frame
new_dataset = pd.concat(new_ds)

# Print the combined DataFrame
print(new_dataset)
#new_dataset.to_csv('append_data.csv', index=False)

    ambient_temperature   humidity  dissolved_oxygen  electrical_conductivity   
0             34.896598  62.753868          6.307200                 9.256473  \
0             34.896598  62.753868          6.307200                 9.256473   
0             34.896598  62.753868          6.307200                 9.256473   
0             34.896598  62.753868          6.307200                 9.256473   
0             34.896598  62.753868          6.307200                 9.256473   
..                  ...        ...               ...                      ...   
88            33.879655  73.508953          7.605051                 8.715915   
88            33.879655  73.508953          7.605051                 8.715915   
88            33.879655  73.508953          7.605051                 8.715915   
88            33.879655  73.508953          7.605051                 8.715915   
88            33.879655  73.508953          7.605051                 8.715915   

    ph_level  temperature  